# Ngày 7 — Prompt engineering: phân loại ticket hỗ trợ

Task: phân loại ticket vào `{Billing, Technical, Account, Other}` bằng OpenAI API.
So sánh **3 biến thể prompt** (zero-shot / few-shot / ràng buộc định dạng) trên cùng bộ ticket có **gold label**.

Cần `OPENAI_API_KEY` trong env (xem Ngày 6).

In [ ]:
import os
import pandas as pd
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Chưa đặt OPENAI_API_KEY trong env!"
client = OpenAI()
MODEL = "gpt-4o-mini"
LABELS = ["Billing", "Technical", "Account", "Other"]

## Bộ ticket mẫu (có nhãn chuẩn)

In [ ]:
tickets = [
    {"text": "Tôi bị tính phí hai lần cho gói tháng này, xin hoàn lại.", "gold": "Billing"},
    {"text": "App cứ bị crash mỗi lần tôi mở phần báo cáo.", "gold": "Technical"},
    {"text": "Tôi quên mật khẩu và không đăng nhập được.", "gold": "Account"},
    {"text": "Cho hỏi giờ làm việc của bộ phận hỗ trợ?", "gold": "Other"},
    {"text": "Hóa đơn tháng trước ghi sai số tiền, cần điều chỉnh.", "gold": "Billing"},
    {"text": "Không nhận được email xác thực để kích hoạt tài khoản.", "gold": "Account"},
]
pd.DataFrame(tickets)

## 3 biến thể prompt — điền vào các hàm dưới

Mỗi hàm nhận `text` (nội dung ticket) và trả về **prompt hoàn chỉnh** gửi cho model.

In [ ]:
def prompt_zero_shot(text):
    # TODO: chỉ mô tả nhiệm vụ + liệt kê nhãn, không ví dụ
    return f"Phân loại ticket sau vào một trong {LABELS}. Chỉ trả về tên nhãn.\nTicket: {text}"

def prompt_few_shot(text):
    # TODO: thêm 2-3 ví dụ mẫu (ticket -> nhãn) trước khi hỏi
    examples = """Ví dụ:
Ticket: \"Thẻ của tôi bị từ chối khi thanh toán\" -> Billing
Ticket: \"Tôi không đăng nhập được vào tài khoản\" -> Account
"""
    return f"Phân loại ticket vào một trong {LABELS}. Chỉ trả về tên nhãn.\n{examples}Ticket: {text} ->"

def prompt_constrained(text):
    # TODO: siết định dạng đầu ra thật chặt
    return (
        f"Bạn là bộ phân loại. Chỉ được in ra ĐÚNG MỘT từ trong danh sách {LABELS}, "
        f"viết hoa chữ đầu, không thêm dấu câu hay giải thích.\nTicket: {text}"
    )

VARIANTS = {"zero_shot": prompt_zero_shot, "few_shot": prompt_few_shot, "constrained": prompt_constrained}

## Hàm phân loại + chạy toàn bộ

In [ ]:
def classify(text, prompt_builder):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt_builder(text)}],
        temperature=0,  # để kết quả ổn định khi so sánh
        max_tokens=5,
    )
    out = resp.choices[0].message.content.strip()
    # chuẩn hóa: lấy nhãn khớp nếu model lỡ thêm chữ thừa
    for lab in LABELS:
        if lab.lower() in out.lower():
            return lab
    return out

rows = []
for t in tickets:
    row = {"text": t["text"], "gold": t["gold"]}
    for name, builder in VARIANTS.items():
        row[name] = classify(t["text"], builder)
    rows.append(row)

result = pd.DataFrame(rows)
result

## Độ chính xác từng biến thể

In [ ]:
for name in VARIANTS:
    acc = (result[name] == result["gold"]).mean()
    print(f"{name:12s}: {acc:.0%}")

## Kết luận

_(Chọn biến thể tốt nhất và **giải thích vì sao** — VD few-shot giúp model bám nhãn hiếm; ràng buộc định dạng giảm output rác.)_